# This code was used to create the ChromaDB for semantic search and doesn't need to be run

In [1]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="science_facts",
    embedding_function=embedding_fn
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
import bz2

articles = []

with bz2.open(
    "simplewiki-latest-pages-articles.xml.bz2",
    "rt",
    encoding="utf-8"
) as f:

    current_title = None
    current_text = []

    for line in f:

        # Article titles start/end with "="
        if line.startswith("=") and line.endswith("=\n"):
            
            # save previous article
            if current_title:
                articles.append({
                    "title": current_title,
                    "text": "".join(current_text)
                })

            current_title = line.strip("=\n ").strip()
            current_text = []

        else:
            current_text.append(line)

# add final article
if current_title:
    articles.append({
        "title": current_title,
        "text": "".join(current_text)
    })

len(articles)

723542

In [12]:
science_keywords = ['physics', 'chemistry', 'biology', 'astronomy', 
                    'molecule', 'atom', 'planet', 'evolution', 'cell',
                    'element', 'energy', 'force', 'species', 'genome']

science_articles = [a for a in articles 
                    if any(kw in a['title'].lower() for kw in science_keywords)]

science_and_long = [a for a in articles 
                    if any(kw in a['title'].lower() for kw in science_keywords)
                    and len(a['text']) >= 200]

In [19]:
import re
import html

def clean_text(text: str) -> str:
    """Clean noisy text while preserving readable content."""
    if not isinstance(text, str):
        return text

    # Decode HTML entities (&amp;, &nbsp;, etc.)
    text = html.unescape(text)

    # Remove citation markers like [1], [citation needed]
    text = re.sub(r'\[[^\]]*\]', '', text)

    # Remove parenthetical pronunciations
    text = re.sub(r'\([^)]*[/\\][^)]*\)', '', text)

    # Replace newlines/tabs with spaces
    text = re.sub(r'[\r\n\t]+', ' ', text)

    # Remove non-printable characters
    text = ''.join(c for c in text if c.isprintable())

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    return text.strip()


def clean_articles(articles):
    """
    Clean all string values in a list of article dictionaries.
    """
    cleaned_articles = []

    for article in articles:
        cleaned_article = {
            key: clean_text(value) if isinstance(value, str) else value
            for key, value in article.items()
        }
        cleaned_articles.append(cleaned_article)

    return cleaned_articles

In [20]:
cleaned_science = clean_articles(science_and_long)
cleaned_science

[{'title': 'Atomic number',
  'text': "] organizes all known chemical elements.]] The number of protons in an atom is called its '']''. Atoms of the same element have the same atomic number. For example, all carbon atoms have six protons, so the atomic number of carbon is six.{{sfn|Flowers et al.|2019|pp=79-85}} Today, 118 elements are known. Depending on how the number is counted, 90 to 94 elements exist naturally on earth. All elements above number 94 have only been made by humans.<ref>{{Cite web|last=McMahon|first=Mary|date=July 27, 2022|title=How Many Elements on the Periodic Table of the Elements Occur Naturally?|url=http://www.allthescience.org/how-many-elements-on-the-periodic-table-of-the-elements-occur-naturally.htm|access-date=August 22, 2022|website=All the Science|language=}}</ref> These elements are organized on the ]."},
 {'title': 'Atomic mass and weight',
  'text': 'Because protons and neutrons have nearly the same ], and the mass of electrons is very small, we can call